## Random Forest Classifyer

In [1]:
# setup

import pandas as pd
import numpy as np
 
from sklearn.datasets import fetch_openml
from sklearn.model_selection import (StratifiedKFold, cross_val_score, cross_val_predict)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, roc_auc_score, confusion_matrix)
 
RANDOM_STATE = 42  


In [2]:
# daten laden

data = fetch_openml(
    data_id=42742,
    as_frame=True,
    parser="auto"
)

X = data.data.copy()
y = data.target.astype(int)

# id muss nicht gedropt werden, da im open-ml datensatz nicht vorhanden; 
# sicherheitshalber assert
assert "id" not in X.columns

print("Form von X:", X.shape)
print("\nKlassenverteilung:")
print(y.value_counts(normalize=True).sort_index())


Form von X: (595212, 57)

Klassenverteilung:
target
0    0.963552
1    0.036448
Name: proportion, dtype: float64


In [3]:
# Column-typen anhand der Suffixe bestimmen

categorical_cols = [
    c for c in X.columns
    if c.endswith("_cat")]

numeric_and_binary_cols = [c for c in X.columns if c not in categorical_cols]

print(f"Kategoriale Features (_cat):    {len(categorical_cols)}")
print(f"Numerische/binäre Features:     {len(numeric_and_binary_cols)}")

Kategoriale Features (_cat):    14
Numerische/binäre Features:     43


In [4]:
# preprocessing

# Numerische und binäre Features: fehlende Werte durch den Median des jeweiligen Trainingsfolds ersetzen.
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Kategoriale Features: fehlende Werte durch die häufigste Kategorie des jeweiligen Trainingsfolds ersetzen + One-Hot-Encoding anwenden.
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

NameError: name 'numeric_cols' is not defined

In [20]:
# cross-Validation für alle Modellvarianten

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)
 

In [21]:
# Random Forest – Variante A

rf_a = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=50,
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE
    ))
])

In [22]:
# Cross-Validation: foldweise ROC-AUC und Gini

auc_scores_a = cross_val_score(
    estimator=rf_a,
    X=X,
    y=y,
    cv=skf,
    scoring="roc_auc",
    n_jobs=1
)

gini_scores_a = 2 * auc_scores_a - 1

print("Random Forest – Variante A")
print("AUC je Fold: ", np.round(auc_scores_a, 4))
print(f"Mittlere AUC: {auc_scores_a.mean():.4f}")
print(f"Std. AUC:     {auc_scores_a.std():.4f}")

print("Gini je Fold:", np.round(gini_scores_a, 4))
print(f"Mittlerer Gini: {gini_scores_a.mean():.4f}")
print(f"Std. Gini:      {gini_scores_a.std():.4f}")

Random Forest – Variante A


NameError: name 'np' is not defined

In [23]:
import numpy as np

print("Random Forest – Variante A")
print("AUC je Fold: ", np.round(auc_scores_a, 4))
print(f"Mittlere AUC: {auc_scores_a.mean():.4f}")
print(f"Std. AUC:     {auc_scores_a.std():.4f}")

print("Gini je Fold:", np.round(gini_scores_a, 4))
print(f"Mittlerer Gini: {gini_scores_a.mean():.4f}")
print(f"Std. Gini:      {gini_scores_a.std():.4f}")

Random Forest – Variante A
AUC je Fold:  [0.6383 0.6343 0.6352 0.6368 0.6305]
Mittlere AUC: 0.6350
Std. AUC:     0.0026
Gini je Fold: [0.2766 0.2687 0.2704 0.2736 0.261 ]
Mittlerer Gini: 0.2700
Std. Gini:      0.0053


In [24]:
# Out-of-Fold-Wahrscheinlichkeiten

oof_proba_a = cross_val_predict(
    estimator=rf_a,
    X=X,
    y=y,
    cv=skf,
    method="predict_proba",
    n_jobs=1
)[:, 1]

oof_auc_a = roc_auc_score(y, oof_proba_a)
oof_gini_a = 2 * oof_auc_a - 1

print("\nOOF-Ergebnis – Random Forest Variante A")
print(f"OOF-AUC:  {oof_auc_a:.4f}")
print(f"OOF-Gini: {oof_gini_a:.4f}")


OOF-Ergebnis – Random Forest Variante A
OOF-AUC:  0.6350
OOF-Gini: 0.2699
